Реализовать с помощью `Numpy` класс `MyMLP`, моделирующий работу полносвязной нейронной сети.

Реализуемый класс должен

1. Поддерживать создание любого числа слоев с любым числом нейронов. Тип инициализации весов не регламентируется.
2. Обеспечивать выбор следующих функции активации в рамках каждого слоя: `ReLU`, `sigmoid`, `linear`.
3. Поддерживать решение задачи классификации и регрессии (выбор соответствующего лосса, в том числе для задачи многоклассовой классификации).
4. В процессе обучения использовать самостоятельно реализованный механизм обратного распространения (вывод формул в формате markdown) для применения градиентного и стохастического градиентного спусков (с выбором размера батча)
5. Поддерживать использование `l1`, `l2` и `l1l2` регуляризаций.

Самостоятельно выбрать наборы данных (классификация и регрессия). Провести эксперименты (различные конфигурации сети: количество слоев, нейронов, функции активации, скорость обучения и тп. — минимум 5 различных конфигураций) и сравнить результаты работы (оценка качества модели + время обучения и инференса) реализованного класса `MyMLP` со следующими моделям (в одинаковых конфигурациях):

*   MLPClassifier/MLPRegressor из sklearn
*   TensorFlow
*   Keras
*   PyTorch

Результат представить в виде .ipynb блокнота, содержащего весь необходимый код и визуализации сравнения реализаций для рассмотренных конфигураций.


In [ ]:
import numpy as np

class MyMLP:
    def __init__(self, layers, activations, loss='mse', lr=0.01, reg=None, reg_lambda=0.0):
        """
        layers: list of neuron counts [input, hidden..., output]
        activations: list of activations per layer (except input)
        loss: 'mse', 'binary_crossentropy', 'categorical_crossentropy'
        reg: None, 'l1', 'l2', 'l1l2'
        """
        self.layers = layers
        self.activations = activations
        self.loss_name = loss
        self.lr = lr
        self.reg = reg
        self.reg_lambda = reg_lambda

        self.weights = []
        self.biases = []

        for i in range(len(layers)-1):
            w = np.random.randn(layers[i], layers[i+1]) * np.sqrt(2/layers[i])
            b = np.zeros((1, layers[i+1]))
            self.weights.append(w)
            self.biases.append(b)

    def _activation(self, z, func):
        if func == 'relu':
            return np.maximum(0, z)
        elif func == 'sigmoid':
            return 1 / (1 + np.exp(-z))
        elif func == 'linear':
            return z

    def _activation_derivative(self, z, func):
        if func == 'relu':
            return (z > 0).astype(float)
        elif func == 'sigmoid':
            s = 1 / (1 + np.exp(-z))
            return s * (1 - s)
        elif func == 'linear':
            return np.ones_like(z)

    def _loss(self, y, y_pred):
        m = y.shape[0]
        if self.loss_name == 'mse':
            return np.mean((y - y_pred) ** 2)
        elif self.loss_name == 'binary_crossentropy':
            return -np.mean(y*np.log(y_pred+1e-8) + (1-y)*np.log(1-y_pred+1e-8))
        elif self.loss_name == 'categorical_crossentropy':
            return -np.mean(np.sum(y*np.log(y_pred+1e-8), axis=1))

    def _loss_derivative(self, y, y_pred):
        m = y.shape[0]
        if self.loss_name == 'mse':
            return (y_pred - y) * 2 / m
        elif self.loss_name == 'binary_crossentropy':
            return (y_pred - y) / m
        elif self.loss_name == 'categorical_crossentropy':
            return (y_pred - y) / m

    def forward(self, X):
        a = X
        activations = [X]
        zs = []

        for w, b, act in zip(self.weights, self.biases, self.activations):
            z = a @ w + b
            a = self._activation(z, act)
            zs.append(z)
            activations.append(a)

        return activations, zs

    def backward(self, X, y):
        activations, zs = self.forward(X)
        grads_w = [None]*len(self.weights)
        grads_b = [None]*len(self.biases)

        delta = self._loss_derivative(y, activations[-1])

        for i in reversed(range(len(self.weights))):
            dz = delta * self._activation_derivative(zs[i], self.activations[i])
            grads_w[i] = activations[i].T @ dz
            grads_b[i] = np.sum(dz, axis=0, keepdims=True)

            if i > 0:
                delta = dz @ self.weights[i].T

        for i in range(len(self.weights)):
            if self.reg == 'l2':
                grads_w[i] += self.reg_lambda * self.weights[i]
            elif self.reg == 'l1':
                grads_w[i] += self.reg_lambda * np.sign(self.weights[i])
            elif self.reg == 'l1l2':
                grads_w[i] += self.reg_lambda * (np.sign(self.weights[i]) + self.weights[i])

        return grads_w, grads_b

    def update(self, grads_w, grads_b):
        for i in range(len(self.weights)):
            self.weights[i] -= self.lr * grads_w[i]
            self.biases[i] -= self.lr * grads_b[i]

    def fit(self, X, y, epochs=100, batch_size=None):
        n = X.shape[0]

        for epoch in range(epochs):
            indices = np.random.permutation(n)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for i in range(0, n, batch_size or n):
                X_batch = X_shuffled[i:i+(batch_size or n)]
                y_batch = y_shuffled[i:i+(batch_size or n)]

                grads_w, grads_b = self.backward(X_batch, y_batch)
                self.update(grads_w, grads_b)

            if epoch % 10 == 0:
                loss = self._loss(y, self.predict(X))
                print(f"Epoch {epoch}, loss: {loss}")

    def predict(self, X):
        a = X
        for w, b, act in zip(self.weights, self.biases, self.activations):
            z = a @ w + b
            a = self._activation(z, act)
        return a



**Формулы производных обратного распространения**

Прямой проход:

$z^l = a^{l-1} W^l + b^l$

$a^l = \sigma(z^l)$

Лосс:

$\frac{\partial L}{\partial a} = \frac{(a-y)}{m}$

Обратное распространение на выходных слоях:

$\delta^L = \frac{\partial L}{\partial a^L} \odot \sigma'(z^L)$

На скрытых слоях:

$\delta^l = (\delta^{l+1} W^{l+1^T}) \odot \sigma'(z^l)$

Градиенты:

$\frac{\partial L}{\partial W^l} = a^{l-1^T} \delta^l$

$\frac{\partial L}{\partial b^l} = \sum \delta^l$

Регуляризация:

$L2: loss + \lambda W  $

$L1: loss + \lambda sign(W)$


In [ ]:
# ==============================
# DATASETS + FRAMEWORK COMPARISON
# ==============================

from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ------------------------------
# 1. CLASSIFICATION DATASET
# ------------------------------

X_cls, y_cls = load_breast_cancer(return_X_y=True)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.2)

scaler_c = StandardScaler()
X_train_c = scaler_c.fit_transform(X_train_c)
X_test_c = scaler_c.transform(X_test_c)

# reshape targets
y_train_c = y_train_c.reshape(-1, 1)
y_test_c = y_test_c.reshape(-1, 1)

# ------------------------------
# 2. REGRESSION DATASET
# ------------------------------

X_reg, y_reg = fetch_california_housing(return_X_y=True)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2)

scaler_r = StandardScaler()
X_train_r = scaler_r.fit_transform(X_train_r)
X_test_r = scaler_r.transform(X_test_r)

# reshape targets
y_train_r = y_train_r.reshape(-1, 1)
y_test_r = y_test_r.reshape(-1, 1)

In [ ]:
# ------------------------------
# 3. TRAIN MyMLP
# ------------------------------

print("=== MyMLP Classification ===")
mlp_cls = MyMLP(
    layers=[X_train_c.shape[1], 32, 16, 1],
    activations=['relu', 'relu', 'sigmoid'],
    loss='binary_crossentropy',
    lr=0.01
)

mlp_cls.fit(X_train_c, y_train_c, epochs=100, batch_size=32)

preds = (mlp_cls.predict(X_test_c) > 0.5).astype(int)
acc = np.mean(preds == y_test_c)
print("Accuracy:", acc)

print("=== MyMLP Regression ===")
mlp_reg = MyMLP(
    layers=[X_train_r.shape[1], 64, 32, 1],
    activations=['relu', 'relu', 'linear'],
    loss='mse',
    lr=0.001
)

mlp_reg.fit(X_train_r, y_train_r, epochs=100, batch_size=32)

preds = mlp_reg.predict(X_test_r)
mse = np.mean((preds - y_test_r) ** 2)
print("MSE:", mse)

=== MyMLP Classification ===
Epoch 0, loss: 0.4757394191765251
Epoch 10, loss: 0.3475088976843797
Epoch 20, loss: 0.27569355789742955
Epoch 30, loss: 0.23230674127983675
Epoch 40, loss: 0.2037346154110625
Epoch 50, loss: 0.18365258387623848
Epoch 60, loss: 0.16857187251856973
Epoch 70, loss: 0.15659350979818706
Epoch 80, loss: 0.14666481639830597
Epoch 90, loss: 0.13834181545226762
Accuracy: 0.956140350877193
=== MyMLP Regression ===
Epoch 0, loss: 0.8389424559559051
Epoch 10, loss: 0.4895305814226657
Epoch 20, loss: 0.42587104461650743
Epoch 30, loss: 0.3959336801463099
Epoch 40, loss: 0.3765750266765553
Epoch 50, loss: 0.3629776039025424
Epoch 60, loss: 0.3522915923916338
Epoch 70, loss: 0.346846497384943
Epoch 80, loss: 0.33788034523902116
Epoch 90, loss: 0.33246900042467514
MSE: 0.3135478580686489


In [ ]:
# ------------------------------
# 4. SKLEARN
# ------------------------------

from sklearn.neural_network import MLPClassifier, MLPRegressor

print("=== sklearn Classification ===")
sk_cls = MLPClassifier(hidden_layer_sizes=(32,16), max_iter=200)
sk_cls.fit(X_train_c, y_train_c.ravel())
print("Accuracy:", sk_cls.score(X_test_c, y_test_c))

print("=== sklearn Regression ===")
sk_reg = MLPRegressor(hidden_layer_sizes=(64,32), max_iter=200)
sk_reg.fit(X_train_r, y_train_r.ravel())
print("MSE:", np.mean((sk_reg.predict(X_test_r) - y_test_r.ravel())**2))

=== sklearn Classification ===


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Accuracy: 0.9736842105263158
=== sklearn Regression ===
MSE: 0.2412015129688734


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
# ------------------------------
# 5. TENSORFLOW / KERAS
# ------------------------------

import tensorflow as tf
from tensorflow import keras

print("=== TensorFlow Classification ===")
model_tf_cls = keras.Sequential([
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model_tf_cls.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_tf_cls.fit(X_train_c, y_train_c, epochs=50, batch_size=32, verbose=0)
print("Accuracy:", model_tf_cls.evaluate(X_test_c, y_test_c, verbose=0)[1])

print("=== TensorFlow Regression ===")
model_tf_reg = keras.Sequential([
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1)
])

model_tf_reg.compile(optimizer='adam', loss='mse')
model_tf_reg.fit(X_train_r, y_train_r, epochs=50, batch_size=32, verbose=0)
preds = model_tf_reg.predict(X_test_r)
print("MSE:", np.mean((preds - y_test_r)**2))

=== TensorFlow Classification ===
Accuracy: 0.9824561476707458
=== TensorFlow Regression ===
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
MSE: 0.2504083294101902


In [ ]:
# ------------------------------
# 6. PYTORCH
# ------------------------------

import torch
import torch.nn as nn
import torch.optim as optim

class TorchMLP(nn.Module):
    def __init__(self, input_dim, hidden, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], output_dim)
        )

    def forward(self, x):
        return self.net(x)

# Classification
print("=== PyTorch Classification ===")
model = TorchMLP(X_train_c.shape[1], [32,16], 1)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

X_t = torch.tensor(X_train_c, dtype=torch.float32)
y_t = torch.tensor(y_train_c, dtype=torch.float32)

for epoch in range(50):
    optimizer.zero_grad()
    out = model(X_t)
    loss = criterion(out, y_t)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    preds = torch.sigmoid(model(torch.tensor(X_test_c, dtype=torch.float32)))
    acc = ((preds > 0.5).numpy() == y_test_c).mean()
    print("Accuracy:", acc)

# Regression
print("=== PyTorch Regression ===")
model = TorchMLP(X_train_r.shape[1], [64,32], 1)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

X_t = torch.tensor(X_train_r, dtype=torch.float32)
y_t = torch.tensor(y_train_r, dtype=torch.float32)

for epoch in range(50):
    optimizer.zero_grad()
    out = model(X_t)
    loss = criterion(out, y_t)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    preds = model(torch.tensor(X_test_r, dtype=torch.float32)).numpy()
    print("MSE:", np.mean((preds - y_test_r)**2))


=== PyTorch Classification ===
Accuracy: 0.9824561403508771
=== PyTorch Regression ===
MSE: 1.396679152839601
